In [1]:
from docx import Document

def read_docx(filename="LuatLaoDong2019.docx"):
    doc = Document(filename)

    paragraphs = []

    for paragraph in doc.paragraphs:
        text = paragraph.text.strip()

        if text:
            paragraphs.append(text)

    return paragraphs

paragraphs = read_docx()

print(f"Total paragraph: {len(paragraphs)}")

for i, p in enumerate(paragraphs[:20], start=1):
    print(f"{i:02d}. {p}")

Total paragraph: 1337
01. BỘ LUẬT
02. LAO ĐỘNG
03. Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam;
04. Quốc hội ban hành Bộ luật Lao động.
05. Chương I
06. NHỮNG QUY ĐỊNH CHUNG
07. Điều 1. Phạm vi điều chỉnh
08. Bộ luật Lao động quy định tiêu chuẩn lao động; quyền, nghĩa vụ, trách nhiệm của người lao động, người sử dụng lao động, tổ chức đại diện người lao động tại cơ sở, tổ chức đại diện người sử dụng lao động trong quan hệ lao động và các quan hệ khác liên quan trực tiếp đến quan hệ lao động; quản lý nhà nước về lao động.
09. Điều 2. Đối tượng áp dụng
10. 1. Người lao động, người học nghề, người tập nghề và người làm việc không có quan hệ lao động.
11. 2. Người sử dụng lao động.
12. 3. Người lao động nước ngoài làm việc tại Việt Nam.
13. 4. Cơ quan, tổ chức, cá nhân khác có liên quan trực tiếp đến quan hệ lao động.
14. Điều 3. Giải thích từ ngữ
15. Trong Bộ luật này, các từ ngữ dưới đây được hiểu như sau:
16. 1. Người lao động là người làm việc cho người sử dụng lao động th

In [2]:
#Regex
import re

RE_CHUONG = re.compile(r"^Chương\s+([IVXLCDM]+)\s*$", re.IGNORECASE)
RE_DIEU = re.compile(r"^Điều\s+(\d+)\.\s*(.+)$", re.IGNORECASE)
RE_KHOAN = re.compile(r"^(\d+)\.\s+(.+)$")
RE_DIEM = re.compile(r"^([a-zđ])\)\s+(.+)$", re.IGNORECASE)

def tokenize_paragraphs(paragraphs):
    tokens = []

    i = 0

    while i < len(paragraphs):

        text = paragraphs[i]

        #CHƯƠNG
        m = RE_CHUONG.match(text)

        if m:
            title = ""

            if i + 1 < len(paragraphs):

                nxt = paragraphs[i + 1]

                if (
                    not RE_CHUONG.match(nxt)
                    and not RE_DIEU.match(nxt)
                    and not RE_KHOAN.match(nxt)
                    and not RE_DIEM.match(nxt)
                ):
                    title = nxt
                    i += 1

            tokens.append({
                "type": "CHUONG",
                "number": m.group(1),
                "title": title,
                "text": text
            })

            i += 1
            continue

        #ĐIỀU
        m = RE_DIEU.match(text)

        if m:
            tokens.append({
                "type": "DIEU",
                "number": m.group(1),
                "title": m.group(2).strip(),
                "text": text
            })

            i += 1
            continue
        
        #KHOẢN
        m = RE_KHOAN.match(text)

        if m:
            tokens.append({
                "type": "KHOAN",
                "number": m.group(1),
                "text": m.group(2).strip()
            })

            i += 1
            continue

        #ĐIỂM
        m = RE_DIEM.match(text)

        if m:
            tokens.append({
                "type": "DIEM",
                "number": m.group(1),
                "text": m.group(2).strip()
            })

            i += 1
            continue

        #TEXT
        tokens.append({
            "type": "TEXT",
            "text": text
        })

        i += 1

    return tokens

In [3]:
#Review
tokens = tokenize_paragraphs(paragraphs)

print(f"Total tokens: {len(tokens)}\n")

for token in tokens[:20]:
    print(token)

Total tokens: 1320

{'type': 'TEXT', 'text': 'BỘ LUẬT'}
{'type': 'TEXT', 'text': 'LAO ĐỘNG'}
{'type': 'TEXT', 'text': 'Căn cứ Hiến pháp nước Cộng hòa xã hội chủ nghĩa Việt Nam;'}
{'type': 'TEXT', 'text': 'Quốc hội ban hành Bộ luật Lao động.'}
{'type': 'CHUONG', 'number': 'I', 'title': 'NHỮNG QUY ĐỊNH CHUNG', 'text': 'Chương I'}
{'type': 'DIEU', 'number': '1', 'title': 'Phạm vi điều chỉnh', 'text': 'Điều 1. Phạm vi điều chỉnh'}
{'type': 'TEXT', 'text': 'Bộ luật Lao động quy định tiêu chuẩn lao động; quyền, nghĩa vụ, trách nhiệm của người lao động, người sử dụng lao động, tổ chức đại diện người lao động tại cơ sở, tổ chức đại diện người sử dụng lao động trong quan hệ lao động và các quan hệ khác liên quan trực tiếp đến quan hệ lao động; quản lý nhà nước về lao động.'}
{'type': 'DIEU', 'number': '2', 'title': 'Đối tượng áp dụng', 'text': 'Điều 2. Đối tượng áp dụng'}
{'type': 'KHOAN', 'number': '1', 'text': 'Người lao động, người học nghề, người tập nghề và người làm việc không có quan hệ 

In [4]:
#Node Model

from itertools import count

node_counter = count(1)


def create_node(node_type, number="", title="", parent=None, path=""):

    return {
        "node_id": next(node_counter),
        "type": node_type,
        "number": number,
        
        "title": title,
        "content": "",

        "parent": parent,
        "children": [],
        "path": path
    }

In [5]:
#Build Tree
LEVEL = {"CHUONG": 1, "DIEU": 2, "KHOAN": 3, "DIEM": 4}

def build_tree(tokens):
    nodes = []

    current_chuong = None
    current_dieu = None
    current_khoan = None
    current_diem = None

    for token in tokens:
        token_type = token["type"]

        #CHUONG
        if token_type == "CHUONG":
            path_item = {
                "type": "CHUONG",
                "number": token["number"],
                "title": token["title"]
            }

            node = create_node(
                node_type="CHUONG",
                number=token["number"],
                title=token["title"],
                parent=None,
                path=[path_item]
            )

            node["level"] = LEVEL["CHUONG"]
            node["content"] = token["title"]
            nodes.append(node)

            current_chuong = node
            current_dieu = None
            current_khoan = None
            current_diem = None

            continue

        #DIEU
        if token_type == "DIEU":
            path_item = {
                "type": "DIEU",
                "number": token["number"],
                "title": token["title"]
            }

            node = create_node(
                node_type="DIEU",
                number=token["number"],
                title=token["title"],
                parent=current_chuong["node_id"],
                path=current_chuong["path"] + [
                    path_item
                ]
            )

            node["level"] = LEVEL["DIEU"]

            current_chuong["children"].append(
                node["node_id"]
            )

            nodes.append(node)

            current_dieu = node
            current_khoan = None
            current_diem = None
            
            continue

        #KHOAN
        if token_type == "KHOAN":

            title = f"Khoản {token['number']}"

            path_item = {
                "type": "KHOAN",
                "number": token["number"],
                "title": title
            }

            node = create_node(
                node_type="KHOAN",
                number=token["number"],
                title=title,
                parent=current_dieu["node_id"],
                path=current_dieu["path"] + [
                    path_item
                ]
            )

            node["level"] = LEVEL["KHOAN"]
            node["content"] = token["text"]
            current_dieu["children"].append(
                node["node_id"]
            )

            nodes.append(node)

            current_khoan = node
            current_diem = None

            continue

        #DIEM
        if token_type == "DIEM":
            title = f"Điểm {token['number']}"

            path_item = {
                "type": "DIEM",
                "number": token["number"],
                "title": title
            }

            node = create_node(
                node_type="DIEM",
                number=token["number"],
                title=title,
                parent=current_khoan["node_id"],
                path=current_khoan["path"] + [
                    path_item
                ]
            )

            node["level"] = LEVEL["DIEM"]
            node["content"] = token["text"]
            current_khoan["children"].append(
                node["node_id"]
            )

            nodes.append(node)
            current_diem = node

            continue

        #Text append
        if token_type == "TEXT":
            target = None
            
            if current_diem is not None:
                target = current_diem

            elif current_khoan is not None:
                target = current_khoan

            elif current_dieu is not None:
                target = current_dieu

            if target is not None:
                if target["content"]:
                    target["content"] += "\n"

                target["content"] += token["text"]

    return nodes

In [6]:
#Preview Tree
nodes = build_tree(tokens)

print(f"Total nodes: {len(nodes)}")

for node in nodes[:10]:
    print(node)

Total nodes: 1169
{'node_id': 1, 'type': 'CHUONG', 'number': 'I', 'title': 'NHỮNG QUY ĐỊNH CHUNG', 'content': 'NHỮNG QUY ĐỊNH CHUNG', 'parent': None, 'children': [2, 3, 8, 18, 26, 39, 52, 57], 'path': [{'type': 'CHUONG', 'number': 'I', 'title': 'NHỮNG QUY ĐỊNH CHUNG'}], 'level': 1}
{'node_id': 2, 'type': 'DIEU', 'number': '1', 'title': 'Phạm vi điều chỉnh', 'content': 'Bộ luật Lao động quy định tiêu chuẩn lao động; quyền, nghĩa vụ, trách nhiệm của người lao động, người sử dụng lao động, tổ chức đại diện người lao động tại cơ sở, tổ chức đại diện người sử dụng lao động trong quan hệ lao động và các quan hệ khác liên quan trực tiếp đến quan hệ lao động; quản lý nhà nước về lao động.', 'parent': 1, 'children': [], 'path': [{'type': 'CHUONG', 'number': 'I', 'title': 'NHỮNG QUY ĐỊNH CHUNG'}, {'type': 'DIEU', 'number': '1', 'title': 'Phạm vi điều chỉnh'}], 'level': 2}
{'node_id': 3, 'type': 'DIEU', 'number': '2', 'title': 'Đối tượng áp dụng', 'content': '', 'parent': 1, 'children': [4, 5, 6,

In [7]:
import json


def export_tree(nodes, output_file="LuatLaoDong2019_tree.json"):

    document = {
        "metadata": {
            "document_name": "LuatLaoDong2019",
            "total_nodes": len(nodes),
            "version": "1.0"
        },
        "nodes": nodes
    }

    with open(output_file, "w", encoding="utf-8") as f:
        json.dump(
            document,
            f,
            ensure_ascii=False,
            indent=2
        )

    print(f"Exported {len(nodes)} nodes -> {output_file}")

In [8]:
node_counter = count(1)
nodes = build_tree(tokens)

export_tree(nodes)

Exported 1169 nodes -> LuatLaoDong2019_tree.json


In [9]:
JSON_FILE = "LuatLaoDong2019_tree.json"

with open(
    JSON_FILE,
    "r",
    encoding="utf-8"
) as f:
    document = json.load(f)

metadata = document["metadata"]
nodes = document["nodes"]

print("Document:")
print(metadata["document_name"])

print(
    "Total nodes:",
    len(nodes)
)

Document:
LuatLaoDong2019
Total nodes: 1169


In [10]:
#Lookup Dictionary
node_index = {
    node["node_id"]: node
    for node in nodes
}


print(
    "Indexed nodes:",
    len(node_index)
)

Indexed nodes: 1169


In [11]:
def get_node(node_id):
    return node_index.get(node_id)

In [12]:
node = get_node(10)

print(node["title"])
print(node["content"][:200])

Khoản 2
Người sử dụng lao động là doanh nghiệp, cơ quan, tổ chức, hợp tác xã, hộ gia đình, cá nhân có thuê mướn, sử dụng người lao động làm việc cho mình theo thỏa thuận; trường hợp người sử dụng lao động là 


In [13]:
def build_navigation_index(nodes):
    navigation = []
    for node in nodes:
        navigation.append({
            "node_id": node["node_id"],
            "level": node["level"],
            "type": node["type"],
            "number": node["number"],
            "title": node["title"],
            "path": node["path"]
        })

    return navigation

In [14]:
navigation_index = build_navigation_index(nodes)

print(
    "Navigation nodes:",
    len(navigation_index)
)

print(
    navigation_index[:3]
)

Navigation nodes: 1169
[{'node_id': 1, 'level': 1, 'type': 'CHUONG', 'number': 'I', 'title': 'NHỮNG QUY ĐỊNH CHUNG', 'path': [{'type': 'CHUONG', 'number': 'I', 'title': 'NHỮNG QUY ĐỊNH CHUNG'}]}, {'node_id': 2, 'level': 2, 'type': 'DIEU', 'number': '1', 'title': 'Phạm vi điều chỉnh', 'path': [{'type': 'CHUONG', 'number': 'I', 'title': 'NHỮNG QUY ĐỊNH CHUNG'}, {'type': 'DIEU', 'number': '1', 'title': 'Phạm vi điều chỉnh'}]}, {'node_id': 3, 'level': 2, 'type': 'DIEU', 'number': '2', 'title': 'Đối tượng áp dụng', 'path': [{'type': 'CHUONG', 'number': 'I', 'title': 'NHỮNG QUY ĐỊNH CHUNG'}, {'type': 'DIEU', 'number': '2', 'title': 'Đối tượng áp dụng'}]}]


In [15]:
#Compact tree for LLM search
def build_compact_tree(nodes):
    node_map = {
        node["node_id"]: node
        for node in nodes
    }

    roots = [
        node
        for node in nodes
        if node["parent"] is None
    ]

    lines = []

    def walk(node, depth=0):
        indent = "    " * depth
        line = (
            f"{indent}"
            f"[{node['node_id']}] "
            f"{node['type']} "
            f"{node['number']} - "
            f"{node['title']}"
        )

        lines.append(line)
        preview = node.get("content", "")

        if preview:
            preview = (
                preview
                .replace("\n", " ")
                .strip()
            )
            
            preview = preview[:80]

            lines.append(
                f"{indent}    Preview: {preview}"
            )

        for child_id in node["children"]:
            child = node_map.get(child_id)
            if child:
                walk(child, depth + 1)

    for root in roots:
        walk(root)

    return "\n".join(lines)

In [16]:
tree_outline = build_compact_tree(nodes)

print(tree_outline[:3000])

[1] CHUONG I - NHỮNG QUY ĐỊNH CHUNG
    Preview: NHỮNG QUY ĐỊNH CHUNG
    [2] DIEU 1 - Phạm vi điều chỉnh
        Preview: Bộ luật Lao động quy định tiêu chuẩn lao động; quyền, nghĩa vụ, trách nhiệm của 
    [3] DIEU 2 - Đối tượng áp dụng
        [4] KHOAN 1 - Khoản 1
            Preview: Người lao động, người học nghề, người tập nghề và người làm việc không có quan h
        [5] KHOAN 2 - Khoản 2
            Preview: Người sử dụng lao động.
        [6] KHOAN 3 - Khoản 3
            Preview: Người lao động nước ngoài làm việc tại Việt Nam.
        [7] KHOAN 4 - Khoản 4
            Preview: Cơ quan, tổ chức, cá nhân khác có liên quan trực tiếp đến quan hệ lao động.
    [8] DIEU 3 - Giải thích từ ngữ
        Preview: Trong Bộ luật này, các từ ngữ dưới đây được hiểu như sau:
        [9] KHOAN 1 - Khoản 1
            Preview: Người lao động là người làm việc cho người sử dụng lao động theo thỏa thuận, đượ
        [10] KHOAN 2 - Khoản 2
            Preview: Người sử dụng lao động là doanh n

In [17]:
print("Characters:", len(tree_outline))
print("Approx tokens:", len(tree_outline.split()))

Characters: 141336
Approx tokens: 26047


In [18]:
#Save navigation index
import json

with open(
    "LuatLaoDong2019_navigation.txt",
    "w",
    encoding="utf-8"
) as f:

    f.write(tree_outline)

print("Saved navigation tree")

Saved navigation tree
